# Three-block DFM planning experiment

Train on **9 problem instances** and evaluate on **9 held-out instances**, all using blocks A, B, C. Each stack below is ordered **bottom to top**. Initial arrangements are disjoint across splits; goals and the domain vocabulary may overlap. No held-out target plans enter training.

Run all cells in order using the project virtual environment. The default CPU-friendly model trains for 50 epochs. This small experiment measures performance; it does not guarantee generalization.


In [1]:
from main import (
    ExperimentConfig, make_problem_splits, build_training_dataset,
    train_model, evaluate_model, save_experiment, print_evaluation,
)

config = ExperimentConfig(horizon=10,max_plans_per_problem=2000,epochs=300,d_model=128,num_layers=4,learning_rate=3e-4)
train_instances, test_instances = make_problem_splits(config.seed,num_probs=100)
for split, instances in (("TRAIN", train_instances), ("HELD OUT", test_instances)):
    print(split)
    for item in instances:
        print(f"  {item['id']}: {item['initial_stacks']} -> {item['goal_stacks']}")


TRAIN
  train_01: (('B', 'A', 'C'),) -> (('B', 'C', 'A'),)
  train_02: (('A', 'B', 'C'),) -> (('A',), ('B', 'C'))
  train_03: (('A', 'B'), ('C',)) -> (('A',), ('C', 'B'))
  train_04: (('A',), ('B', 'C')) -> (('B',), ('C', 'A'))
  train_05: (('C', 'B', 'A'),) -> (('A', 'B', 'C'),)
  train_06: (('A', 'C'), ('B',)) -> (('A', 'B', 'C'),)
  train_07: (('B',), ('C', 'A')) -> (('A',), ('C', 'B'))
  train_08: (('B', 'C', 'A'),) -> (('A', 'C', 'B'),)
  train_09: (('C', 'A', 'B'),) -> (('A',), ('B', 'C'))
  train_10: (('B', 'A'), ('C',)) -> (('C', 'A', 'B'),)
  train_11: (('C', 'A', 'B'),) -> (('A', 'C', 'B'),)
  train_12: (('A',), ('C', 'B')) -> (('B',), ('C', 'A'))
  train_13: (('A', 'C', 'B'),) -> (('C', 'A', 'B'),)
  train_14: (('A',), ('B', 'C')) -> (('A', 'C', 'B'),)
  train_15: (('B', 'A', 'C'),) -> (('A', 'B', 'C'),)
  train_16: (('C', 'B', 'A'),) -> (('B',), ('C', 'A'))
  train_17: (('A',), ('B',), ('C',)) -> (('A',), ('B', 'C'))
  train_18: (('A', 'C', 'B'),) -> (('A',), ('C', 'B'))
  

## Build the training dataset

Enumerate goal-satisfying plans up to 8 actions, keep at most 128 per instance using a fixed seed, and verify every retained target with the plan validator. A shared codec gives every problem the same vocabulary and tensor shapes. Only the training instances are passed to this step.


In [2]:
dataset, codec, counts = build_training_dataset(train_instances, config)
for row in counts:
    print(f"{row['id']}: {row['training_plans']} retained / {row['enumerated_plans']} valid plans")
print(f"Total training examples: {len(dataset)}")
print(f"Context shape: {tuple(dataset[0]['context'].shape)}")
print(f"Plan shape: {tuple(dataset[0]['plan'].shape)}")


NOTE: To disable printing of planning engine credits, add this line to your code: `up.shortcuts.get_environment().credits_stream = None`
  *** Credits ***
  * In operation mode `Compiler` at line 180 of `/home/itay.bryt/Documents/Planning_DFM/trajectory_generator.py`, you are using the following planning engine:
  * Engine name: Fast Downward
  * Developers:  Uni Basel team and contributors (cf. https://github.com/aibasel/downward/blob/main/README.md)
  * Description: Fast Downward is a domain-independent classical planning system.

  *** Credits ***
  * In operation mode `Compiler` at line 180 of `/home/itay.bryt/Documents/Planning_DFM/trajectory_generator.py`, you are using the following planning engine:
  * Engine name: Fast Downward
  * Developers:  Uni Basel team and contributors (cf. https://github.com/aibasel/downward/blob/main/README.md)
  * Description: Fast Downward is a domain-independent classical planning system.

  *** Credits ***
  * In operation mode `Compiler` at line 

## Train the conditional DFM

The existing `train_dfm` loop learns to reconstruct clean plans from partially masked plans, conditioned on the initial state and goal. `history` records loss and masked-token accuracy for each epoch. These are training metrics, not planning success rates.


In [3]:
model, history = train_model(dataset, codec, config)


/home/itay.bryt/Documents/Planning_DFM/model.py:263: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.transformer = nn.TransformerEncoder(


Training on cuda.
Epoch 001 | loss=0.4802 | masked accuracy=0.5416
Epoch 002 | loss=0.3139 | masked accuracy=0.7211
Epoch 003 | loss=0.2520 | masked accuracy=0.7747
Epoch 004 | loss=0.2341 | masked accuracy=0.7848
Epoch 005 | loss=0.2293 | masked accuracy=0.7905
Epoch 006 | loss=0.2213 | masked accuracy=0.7961
Epoch 007 | loss=0.2181 | masked accuracy=0.7988
Epoch 008 | loss=0.2133 | masked accuracy=0.8026
Epoch 009 | loss=0.2132 | masked accuracy=0.8049
Epoch 010 | loss=0.2094 | masked accuracy=0.8081
Epoch 011 | loss=0.2076 | masked accuracy=0.8096
Epoch 012 | loss=0.2042 | masked accuracy=0.8137
Epoch 013 | loss=0.1997 | masked accuracy=0.8159
Epoch 014 | loss=0.1957 | masked accuracy=0.8209
Epoch 015 | loss=0.1979 | masked accuracy=0.8194
Epoch 016 | loss=0.1901 | masked accuracy=0.8256
Epoch 017 | loss=0.1913 | masked accuracy=0.8245
Epoch 018 | loss=0.1879 | masked accuracy=0.8282
Epoch 019 | loss=0.1908 | masked accuracy=0.8256
Epoch 020 | loss=0.1875 | masked accuracy=0.8273
Ep

## Evaluate on the nine unseen problems

Generate 32 raw plans per problem with 64 DFM sampling steps. Strict validation checks formatting, action applicability, and final goal satisfaction. All samples count in the denominator, including malformed plans. There is no repair or reference-plan substitution.

- **Satisfaction rate:** goal-satisfying samples / all 288 generated samples.
- **Executable rate:** correctly formatted, executable samples / all samples (the goal may still be unmet).
- **Problems solved:** held-out problems with at least one satisfying sample / 9. This depends on the sampling budget and is not single-sample accuracy.


In [4]:
evaluation = evaluate_model(model, test_instances, codec, config)
print_evaluation(evaluation)
print(f"Executable rate: {evaluation['summary']['executable_rate']:.1%}")



Held-out evaluation (raw samples, strict plan validation)
Problem        Executable   Satisfying   Satisfaction
test_01          30/32        30/32            93.8%
test_02          30/32        29/32            90.6%
test_03          27/32        27/32            84.4%
test_04          28/32        28/32            87.5%
test_05          31/32        26/32            81.2%
test_06          25/32        25/32            78.1%
test_07          30/32        30/32            93.8%
test_08          28/32        27/32            84.4%
test_09          30/32        30/32            93.8%
test_10          28/32        27/32            84.4%
test_11          29/32        28/32            87.5%
test_12          25/32        24/32            75.0%
test_13          30/32        30/32            93.8%
test_14          26/32        26/32            81.2%
test_15          27/32        27/32            84.4%
test_16          28/32        27/32            84.4%
test_17          28/32        28/32    

## Save and inspect results

Save the dataset, trained weights, exact problem split, training history, and every generated token tensor with its validation outcome. Failure examples help distinguish formatting issues from action-precondition or goal failures. Zero observed successes means this run did not produce a solution; it does not imply the planning problems are unsatisfiable.


In [5]:
save_experiment(
    config, train_instances, test_instances, dataset, codec, counts,
    model, history, evaluation,
)
print(f"Artifacts saved to {config.output_dir}")
for sample in [s for s in evaluation["samples"] if not s["solves_problem"]][:5]:
    print(f"{sample['id']} sample {sample['sample']}: {sample['reason']}")


Artifacts saved to artifacts/three_blocks
test_01 sample 0: Action pick_up has unexpected arguments: ['pick_up', 'C', 'A']
test_01 sample 1: PAD encountered before END at row 0.
test_01 sample 2: Action pick_up has unexpected arguments: ['pick_up', 'B', 'A']
test_01 sample 3: Action put_down has unexpected arguments: ['put_down', 'C', 'A']
test_01 sample 4: Missing/invalid argument 1 of action stack: ['stack', 'C', '<NONE>']


In [5]:
def success_rate_stats(evaluation):
    samples_by_problem = {}

    for sample in evaluation["samples"]:
        samples_by_problem.setdefault(sample["id"], []).append(
            bool(sample["solves_problem"])
        )

    if not samples_by_problem:
        print("No evaluation samples found.")
        return {"rates": {}, "mean": 0.0, "variance": 0.0}

    rates = {
        problem_id: sum(results) / len(results)
        for problem_id, results in samples_by_problem.items()
    }
    mean_rate = sum(rates.values()) / len(rates)
    variance = sum((rate - mean_rate) ** 2 for rate in rates.values()) / len(rates)

    print("Success rates by problem")
    print("-" * 36)
    for problem_id, rate in sorted(rates.items()):
        print(f"{problem_id:<16} {rate:>7.1%}")
    print("-" * 36)
    print(f"{'Mean rate':<16} {mean_rate:>7.1%}")
    print(f"{'Variance':<16} {variance:>7.6f}")

    return {"rates": rates, "mean": mean_rate, "variance": variance}


success_stats = success_rate_stats(evaluation)

Success rates by problem
------------------------------------
test_01            93.8%
test_02            90.6%
test_03            84.4%
test_04            87.5%
test_05            81.2%
test_06            78.1%
test_07            93.8%
test_08            84.4%
test_09            93.8%
test_10            84.4%
test_11            87.5%
test_12            75.0%
test_13            93.8%
test_14            81.2%
test_15            84.4%
test_16            84.4%
test_17            87.5%
test_18            96.9%
test_19            93.8%
test_20            81.2%
------------------------------------
Mean rate          86.9%
Variance         0.003477
